# Week 5 — Trees, Forests, Boosting, and Tuning

**Focus:** What additional structure can flexible trees learn, and what evidence supports their use?

*Live-coding notebook*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/smc77/uc_finmlai/blob/main/lectures/week05/week5_demos.ipynb)

## What this notebook is about

Trees can represent interactions and thresholds that a single line cannot. That flexibility is useful only when the structure repeats outside the data that selected it. This notebook follows six jobs:

**Represent → Control → Average → Correct → Audit → Test**

You will use XOR to expose a representational difference, watch tree depth trade training fit for held-out performance, see why trees do not extrapolate, average trees in forests, build boosting stage by stage, select an early-stopping round on validation, and inspect how a leaked proxy can be amplified.

By the end, you should be able to:

- explain what a tree represents that a line cannot;
- distinguish interpolation from extrapolation;
- describe forests as averaging and boosting as sequential correction;
- compare impurity importance with held-out permutation importance; and
- treat depth, features, learning rate, and stopping round as part of the search.

The synthetic examples reveal known mechanisms. The market companion is empirical; the credit companion is simulated. Neither ranks model families universally.

### Course context

For a concise review and the optional textbook route, see [Week 5 summary](https://github.com/smc77/uc_finmlai/blob/main/lectures/week05/week5_summary.md) and [Week 5 reading guide](https://github.com/smc77/uc_finmlai/blob/main/lectures/week05/reading.md). The notebook itself is designed to remain understandable without them.

## How to use this notebook

At the first break, run any **Setup** cell and then the **Imports** cell. If there is no Setup cell, begin with Imports. After that, use the slide cue to jump to the named demo; you do not need to rerun the whole notebook at every break. To check the whole notebook from a clean start, use **Runtime → Run all**.

Each demo follows the same rhythm: predict what the output should show, run the cell, and follow the task immediately underneath it. An **After you run** cell is editable: double-click it, replace `[write here]`, and press Shift-Enter. When an editable research-record entry appears, its completed comparison sits below it in a collapsed box—write your version before opening that box. This notebook is generated from the same source code the lecturer runs live.

## Jump to a demonstration

Both links jump within *this* notebook — neither opens a new tab. Use **Colab** when the notebook is open in Colab, **Jupyter** when it is open in Jupyter or rendered to HTML. You do not need to scroll through or rerun the whole notebook.

> If the **Jupyter** links do nothing in JupyterLab or Notebook 7, that is the windowed renderer, not a broken link: cells outside the viewport are not in the page, so there is no anchor to scroll to. Set *Settings → Settings Editor → Notebook → Windowing mode* to `defer` (or `none`) and they work.

### Deck A

<ul>
<li>Demo 1 — XOR: a line cannot, a tree can — <a href="#Demo-1-%E2%80%94-XOR%3A-a-line-cannot,-a-tree-can">Jupyter</a> · <a href="#scrollTo=md-78c5b76f2ca4">Colab</a></li>
<li>Demo 2 — Tree overfitting vs. depth — <a href="#Demo-2-%E2%80%94-Tree-overfitting-vs.-depth">Jupyter</a> · <a href="#scrollTo=md-15610279d250">Colab</a></li>
<li>Demo 3 — No extrapolation: tree flatlines outside training range — <a href="#Demo-3-%E2%80%94-No-extrapolation%3A-tree-flatlines-outside-training-range">Jupyter</a> · <a href="#scrollTo=md-f0969bbe613d">Colab</a></li>
<li>Demo 4 — Single tree vs. random forest (held-out ranking) — <a href="#Demo-4-%E2%80%94-Single-tree-vs.-random-forest-(held-out-ranking)">Jupyter</a> · <a href="#scrollTo=md-be17209b4407">Colab</a></li>
<li>Demo 5 — Impurity vs. permutation importance — <a href="#Demo-5-%E2%80%94-Impurity-vs.-permutation-importance">Jupyter</a> · <a href="#scrollTo=md-22c590cb7488">Colab</a></li>
</ul>

### Deck B

<ul>
<li>Demo 6 — Gradient boosting: stagewise build, learning rate, early stopping — <a href="#Demo-6-%E2%80%94-Gradient-boosting%3A-stagewise-build,-learning-rate,-early-stopping">Jupyter</a> · <a href="#scrollTo=md-6ad0829eb82f">Colab</a></li>
<li>Demo 7 — Early stopping: find the round that minimizes validation MSE — <a href="#Demo-7-%E2%80%94-Early-stopping%3A-find-the-round-that-minimizes-validation-MSE">Jupyter</a> · <a href="#scrollTo=md-0386cd24083a">Colab</a></li>
<li>Real-data companion — linear, forest, and boosting forecasts on one market sample — <a href="#Real-data-companion-%E2%80%94-linear,-forest,-and-boosting-forecasts-on-one-market-sample">Jupyter</a> · <a href="#scrollTo=md-320adef803f5">Colab</a></li>
<li>Demo 8 — The leakage amplifier — <a href="#Demo-8-%E2%80%94-The-leakage-amplifier">Jupyter</a> · <a href="#scrollTo=md-265384ee3143">Colab</a></li>
<li>Demo 9 — Freeze the stopping rule, then test once — <a href="#Demo-9-%E2%80%94-Freeze-the-stopping-rule,-then-test-once">Jupyter</a> · <a href="#scrollTo=md-ffb78dd5a632">Colab</a></li>
<li>Simulated companion — credit scoring on held-out borrowers — <a href="#Simulated-companion-%E2%80%94-credit-scoring-on-held-out-borrowers">Jupyter</a> · <a href="#scrollTo=md-f2d6a2c90f52">Colab</a></li>
</ul>

## Read the code: rules, averages, and corrections

A **decision tree** asks a series of yes/no questions. Its **leaves** are the final groups that supply predictions. A **random forest** averages many randomized trees. **Boosting** adds small corrections in sequence.

Start with the question above each cell, predict one result, then run and explain the output. `X` is a table of inputs: rows are cases, columns are features. `y` contains one outcome per row. `.fit(X, y)` learns from those examples; `.predict(X_new)` applies the fitted rule to other cases. `[:, 1]` selects the probability of class 1 from `.predict_proba(...)`.

**Run order:** restart the kernel and run from the top to reproduce the recorded teaching draw. Demos 1–7 share a seeded random-number stream; rerunning an earlier simulation changes later draws. Demos 6, 7, and 9 share one prediction problem. Demo 8 has a separate audit sample so it cannot reveal Demo 9's final test. The optional market example uses separate variable names.

**Know which experiment you are running:** most cells generate independent simulated rows, with no calendar or changing population. Holding out rows teaches fitting, selection, and testing, but does not establish performance on a future market period. Demo 3 has a time-indexed trend; the market companion has actual dates. Simulated credit borrowers are not an observed later cohort.

## Your Week 5 practice research record

Complete the editable entry immediately below each demonstration. **Do not write in this overview.** These entries stay in the weekly notebook and are not a separate graded submission; the final-project record begins in Week 6.

Flexible models make the search itself part of the result. Record the complete comparison, not only the selected model.

### The six lecture breaks

1. **Represent — Demo 1:** compare the linear and tree function classes on the same XOR rows and state the narrow representation claim.
2. **Control — Demos 2–3:** record the depth path and test peak, then distinguish interpolation inside the training range from extrapolation beyond it.
3. **Average — Demos 4–5:** compare one tree with a forest on identical independent rows, then contrast training impurity importance with held-out permutation importance.
4. **Correct — Demos 6–7:** follow the validation path across learning rates and record the validation-selected stopping round rather than the maximum cap.
5. **Audit — Demo 8:** name the planted target proxy, compare admissible and contaminated audit-sample rank correlations, and record the lineage check that would reject it.
6. **Test — Demo 9:** freeze the validation-selected boosting stage, refit through the validation boundary, and test it once beside the development-mean baseline.

The real-market companion after Demo 7 and the credit companion after Demo 9 are optional and have their own entries. For every result, state one narrow claim and one limitation.

### Imports

> **Run this once before any demo.** It loads the packages and helper functions used below; there is no result to interpret in this cell.

In [ ]:
from io import BytesIO, StringIO
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import roc_auc_score, accuracy_score, brier_score_loss
from sklearn.inspection import permutation_importance

# Restart the kernel and run all cells for the recorded teaching draw.
# Demos 1–7 share this stream; Demos 8 and the credit companion have their own.
# Most samples below are IID (independent rows), not dated market cohorts.
rng = np.random.default_rng(42)


def course_csv(relative_path):
    """Read bundled Fama-French data locally, or its public source in Colab."""
    for root in (Path.cwd(), *Path.cwd().parents):
        local = root / relative_path
        if local.exists():
            return pd.read_csv(local), str(local)
    archives = {
        "datasets/famafrench/ff_factors_daily.csv": "F-F_Research_Data_Factors_daily_CSV.zip",
        "datasets/famafrench/ff_12industry_daily.csv": "12_Industry_Portfolios_daily_CSV.zip",
    }
    archive = archives[relative_path]
    url = f"https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/{archive}"
    with urlopen(url) as response, ZipFile(BytesIO(response.read())) as zipped:
        lines = zipped.read(zipped.namelist()[0]).decode("utf-8").splitlines()
    header_row = next(i for i, line in enumerate(lines) if line.startswith(","))
    rows = []
    for line in lines[header_row + 1:]:
        first = line.split(",", 1)[0].strip()
        if len(first) == 8 and first.isdigit():
            rows.append(line)
        elif rows:
            break
    frame = pd.read_csv(StringIO("\n".join([lines[header_row], *rows])))
    return frame.rename(columns={frame.columns[0]: "date"}), url


def save_lecture_figure(fig, filename):
    """Save a notebook figure when the course repository is available."""
    for root in (Path.cwd(), *Path.cwd().parents):
        output = root / "lectures" / "week05" / "figures"
        if output.exists():
            target = output / filename
            fig.savefig(target, dpi=180, bbox_inches="tight", facecolor="white")
            print(f"saved lecture figure: {target}")
            return

### Demo 1 — XOR: a line cannot, a tree can

> **Break cue:** Deck A, after recording segment S1. Read the task on the preceding slide, predict what the output should show, then run the cell. Fill in the **research-record entry** below the output — the fields are named for you, and a completed example sits under it once you have written yours.

### One question, then a second question

`(X[:, 0] > 0) ^ (X[:, 1] > 0)` creates **exclusive-or (XOR)**: the outcome is 1 when exactly one input is positive. The `^` operator means “one or the other, but not both.” `max_depth=1` allows just one question on a path; depth 2 can ask about the second feature after the first.

`DecisionTreeClassifier(...).fit(X, y)` searches for locally useful cuts. `.predict(grid)` colors the regions implied by those cuts. The displayed `.score(X, y)` is **training accuracy**: the fraction of these same examples classified correctly. A model family can represent a pattern that greedy fitting does not recover perfectly from this sample.

**Predict, then check:** can a single vertical or horizontal cut separate all four XOR regions? Trace the point `(1, -1)` through two questions. Explain why perfect training accuracy would still need an independent test.

In [ ]:
N = 1000
X = rng.standard_normal((N, 2))
y = ((X[:, 0] > 0) ^ (X[:, 1] > 0)).astype(int)

logit = LogisticRegression().fit(X, y)
print(f"Logistic training accuracy: {logit.score(X, y):.3f}")
depth_models = {
    depth: DecisionTreeClassifier(max_depth=depth, random_state=0).fit(X, y)
    for depth in (1, 2, 4)
}
for depth, fitted_tree in depth_models.items():
    print(f"Tree(depth={depth}) training accuracy: {fitted_tree.score(X, y):.3f}")

axis = np.linspace(-3, 3, 180)
xx, yy = np.meshgrid(axis, axis)
grid = np.column_stack([xx.ravel(), yy.ravel()])
fig, axes = plt.subplots(1, 3, figsize=(11, 3.3), sharex=True, sharey=True)
for ax, (depth, fitted_tree) in zip(axes, depth_models.items()):
    surface = fitted_tree.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, surface, levels=[-0.5, 0.5, 1.5], alpha=0.25,
                colors=["tab:blue", "tab:orange"])
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=8, alpha=0.45)
    ax.set_title(f"max_depth={depth}")
    ax.set_xlabel("feature 1")
axes[0].set_ylabel("feature 2")
fig.suptitle("The tree family can represent XOR; greedy fitting chooses the partition")
fig.tight_layout()
save_lecture_figure(fig, "demo1_xor_partitions.png")
plt.show()
# Expected: the line remains near chance. Greater depth makes the XOR partition
#           available, but the exact greedy path is a separate estimation question.

#### Your research-record entry

*Double-click this cell to edit it, then press Shift-Enter to render.*

| FIELD | YOUR ENTRY |
|:--|:--|
| known truth | `[how the outcome is generated]` |
| fixed comparison | `[models, rows, and features held the same]` |
| evidence | `[logistic and depth 1, 2, and 4 accuracy values]` |
| narrow claim | `[what the representation comparison supports]` |
| limitation | `[what training accuracy and this simulation cannot establish]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

**Completed representation entry**

| FIELD | EXAMPLE ENTRY |
|:--|:--|
| known truth | 1,000 simulated rows; the outcome is 1 when exactly one of two features is positive |
| fixed comparison | logistic regression and trees of depths 1, 2, and 4 receive the same two features and same rows |
| evidence | logistic accuracy **0.522**; tree accuracies **0.526**, **0.789**, and **1.000** as depth rises |
| narrow claim | the tree can represent this threshold interaction directly, while the unexpanded linear log-odds model cannot |
| limitation | training accuracy does not establish generalization, and this clean XOR simulation is not market evidence |

The comparison isolates representation. It does not yet tell us whether the
selected tree will generalize to noisy later data.

</details>

### Mini-project: draw the partition

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

Run **Demo 1** in `week5_demos.ipynb`.

1. Open the Week 5 tree notebook.
2. Fit depths 1, 2, and 4 to exclusive-or (XOR): exactly one input is positive.
3. Draw each decision surface.
4. Write one sentence separating **representation** from **search**.

**Research record:** the function class and its geometry; the depths compared;
what the representation can express, and what the search actually found.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 2 — Tree overfitting vs. depth

> **Break cue:** Deck A, after recording segment S2. Read the task on the preceding slide, predict what the output should show, then run the cell. Fill in the **research-record entry** below the output — the fields are named for you, and a completed example sits under it once you have written yours.

### Choose a size before revealing the test

Slices `[:750]`, `[750:1500]`, and `[1500:]` assign independent rows to training, validation, and testing. Every candidate is fitted only on `Xtr, ytr`. `np.argmax(va_acc) + 1` selects the depth with highest validation accuracy; `+ 1` converts a zero-based list position to a depth starting at 1. A tie keeps the first, shallower depth.

`np.vstack` combines input rows and `np.concatenate` combines outcomes for a **refit** at that fixed depth. Only then is the test scored. The red and blue lines show training and validation; the single green star shows the refitted model's test result. There is no test curve to browse while choosing depth.

**Predict, then check:** which curve should generally keep improving as depth grows? Explain why the green star need not equal the selected blue point: both the fitted sample and evaluated sample changed.

In [ ]:
N = 2250
X = rng.standard_normal((N, 4))
logits = 0.6 * X[:, 0] * X[:, 1] - 0.4 * np.abs(X[:, 2]) + rng.standard_normal(N) * 1.5
y = (logits > 0).astype(int)
ntr, nval = 750, 750
Xtr, Xva, Xte = X[:ntr], X[ntr:ntr + nval], X[ntr + nval:]
ytr, yva, yte = y[:ntr], y[ntr:ntr + nval], y[ntr + nval:]

depths = range(1, 16)
tr_acc, va_acc = [], []
for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=0).fit(Xtr, ytr)
    tr_acc.append(m.score(Xtr, ytr))
    va_acc.append(m.score(Xva, yva))

selected_depth = int(np.argmax(va_acc)) + 1
selected_tree = DecisionTreeClassifier(max_depth=selected_depth, random_state=0).fit(
    np.vstack([Xtr, Xva]), np.concatenate([ytr, yva])
)
selected_test_accuracy = selected_tree.score(Xte, yte)
print(f"Validation-selected depth: {selected_depth}")
print(f"Validation accuracy at selected depth: {va_acc[selected_depth - 1]:.3f}")
print(f"Test accuracy after refit: {selected_test_accuracy:.3f}")
print(f"Depth-15 training accuracy: {tr_acc[-1]:.3f}")

plt.figure(figsize=(7, 3.5))
plt.plot(list(depths), tr_acc, label="train", color="red")
plt.plot(list(depths), va_acc, label="validation", color="tab:blue")
plt.scatter([selected_depth], [selected_test_accuracy], color="green", marker="*",
            s=130, zorder=5, label="test after selected-depth refit")
plt.axvline(selected_depth, ls="--", color="black", lw=0.8,
            label=f"validation choice={selected_depth}")
plt.xlabel("max_depth"); plt.ylabel("accuracy")
plt.title("Single tree: validation selects depth; test evaluates it")
plt.legend(); plt.tight_layout()
save_lecture_figure(plt.gcf(), "demo2_depth_selection.png")
plt.show()
# Expected: training fit keeps rising. Validation selects a comparatively shallow
#           tree; the third block assesses that already-selected procedure.

#### Your research-record entry

*Double-click this cell to edit it, then press Shift-Enter to render.*

| FIELD | YOUR ENTRY |
|:--|:--|
| design | `[rows, features, signal, and noise]` |
| boundary | `[training, validation, and test rows plus depths compared]` |
| path | `[selected depth plus training, validation, and test accuracy]` |
| selected complexity | `[which part of the path you would choose and why]` |
| narrow claim | `[what the gap supports]` |
| limitation | `[what one simulation and split cannot establish]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

**Completed depth-path entry**

| FIELD | EXAMPLE ENTRY |
|:--|:--|
| design | 2,250 simulated rows, four features, a planted nonlinear interaction, and substantial noise |
| boundary | 750 rows train, the next 750 select depth, and the final 750 test the already-selected procedure |
| path | validation selects depth **1** at accuracy **0.624**; after refitting, test accuracy is **0.585**; depth 15 has training accuracy **0.992**; no test-depth curve is inspected |
| selected complexity | the validation-selected depth, not the deepest tree or the best training score |
| narrow claim | in this controlled sample, extra depth eventually fits details that do not generalize to the third block |
| limitation | one split and one simulation do not identify a universally best depth |

The whole path is evidence. Reporting only the selected depth would hide how
much complexity was searched.

</details>

### Demo 3 — No extrapolation: tree flatlines outside training range

> **Break cue:** Deck A, after recording segment S2. Read the task on the preceding slide, predict what the output should show, then run the cell. Fill in the **research-record entry** below the output — the fields are named for you, and a completed example sits under it once you have written yours.

### Follow the last leaf past the boundary

`np.arange(...)` makes a time index. The known curve is a rising line plus `2 * sin(t / 20)`; this cycle has period `40π`, about 126 time steps. Only the first 250 noisy observations fit the models. `DecisionTreeRegressor(max_depth=5)` stores a constant average in each leaf; `LinearRegression` learns a slope and intercept.

Both `.predict(x)` calls include inputs beyond training support. The tree sends those farther points to the same edge leaf; the line extends its slope. The grey continuation is the simulator's known noiseless curve, not future observations available during fitting. This is a separate example from Demo 2, whose target was binary.

**Predict, then check:** compare tree predictions at time 300 and 360. Explain why the line's ability to continue upward does not prove its forecast is correct.

In [ ]:
T_train, T_extra = 250, 120
x = np.arange(T_train + T_extra).reshape(-1, 1).astype(float)
# Trending series
true = 0.1 * x.flatten() + np.sin(x.flatten() / 20) * 2
y_obs = true[:T_train] + rng.standard_normal(T_train) * 0.5

tree = DecisionTreeRegressor(max_depth=5, random_state=0).fit(x[:T_train], y_obs)
lin = LinearRegression().fit(x[:T_train], y_obs)

pred_tree = tree.predict(x)
pred_lin = lin.predict(x)

plt.figure(figsize=(8, 3.5))
plt.plot(x.flatten(), true, color="grey", lw=0.8, label="true")
plt.axvline(T_train, ls="--", color="k", lw=0.8)
plt.plot(x.flatten(), pred_tree, color="red", label="tree")
plt.plot(x.flatten(), pred_lin, color="blue", ls="--", label="linear")
plt.legend(); plt.xlabel("t"); plt.ylabel("y")
plt.title("Tree (red) flatlines beyond training range; line extrapolates")
plt.tight_layout()
save_lecture_figure(plt.gcf(), "demo3_tree_extrapolation.png")
plt.show()
# Expected: the tree's prediction is flat outside the training range. It cannot
#           extrapolate; a linear fit can.

#### Your research-record entry

*Double-click this cell to edit it, then press Shift-Enter to render.*

| FIELD | YOUR ENTRY |
|:--|:--|
| target and range | `[object plus observed and extrapolation ranges]` |
| models | `[models fit on identical rows]` |
| evidence | `[what each prediction does beyond training]` |
| mechanism | `[why the tree behaves this way]` |
| narrow claim | `[what this reveals about the function class]` |
| limitation | `[what it does not say about target choice or forecast truth]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

**Completed extrapolation entry**

| FIELD | EXAMPLE ENTRY |
|:--|:--|
| target and range | a noisy trending level observed for times 0–249; predictions extend through time 369 |
| models | a depth-5 regression tree and ordinary linear regression fit on the same first 250 observations |
| evidence | the tree makes stepwise predictions in range and stays flat beyond the largest training value; the line continues its fitted slope |
| mechanism | a tree predicts with averages stored in existing leaves and creates no new leaf beyond the observed split thresholds |
| narrow claim | ordinary regression trees do not extrapolate a trend outside their training feature range |
| limitation | the line's ability to extrapolate does not make its extrapolation correct, and a level may still be the right target for some decisions |

This is a statement about function classes, not a command to transform every
financial target into a return.

</details>

### Mini-project: select depth, then test

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

Run **Demos 2–3** in `week5_demos.ipynb`.

1. Fit depths 1 through 15 on the training block.
2. Select depth on the separate validation block.
3. Freeze the choice and score the next block.
4. In Demo 3, fit a separate trending-level example and inspect extrapolation.


Demo 2 uses independent simulated rows; Demo 3 uses a time-indexed trend.

**Research record:** the depth grid searched; the validation selection
rule; the frozen score on the next block; why the separate level example becomes flat beyond training support.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 4 — Single tree vs. random forest (held-out ranking)

> **Break cue:** Deck A, after recording segment S3. Read the task on the preceding slide, predict what the output should show, then run the cell. Fill in the **research-record entry** below the output — the fields are named for you, and a completed example sits under it once you have written yours.

### Average different answers

`n_estimators=300` fits 300 trees. Each sees a bootstrap sample, drawn with replacement, and a random subset of candidate features at each split. The forest averages class probabilities. `oob_decision_function_[:, 1]` predicts each training case using only trees that omitted it; OOB means **out of bag**.

`roc_auc_score(yte, probability)` measures ranking: 0.5 is chance ordering, 1 is perfect ordering. Compare the single tree and forest on identical independent holdout rows. The OOB AUC uses training rows and is a separate diagnostic. This one fit shows a ranking comparison, not an estimate of prediction variance across repeated training samples. With very few trees some rows can lack OOB predictions.

**Predict, then check:** average probabilities 0.31, 0.44, 0.27, and 0.38 by hand. Explain why random omission would not protect a dated forecast from overlapping future labels.

In [ ]:
N = 3000
X = rng.standard_normal((N, 6))
score = (0.6 * X[:, 0] * X[:, 1] - 0.5 * np.abs(X[:, 2])
         + 0.4 * X[:, 3] + rng.standard_normal(N) * 1.2)
y = (score > 0).astype(int)
ntr = N // 2
Xtr, Xte, ytr, yte = X[:ntr], X[ntr:], y[:ntr], y[ntr:]

tree = DecisionTreeClassifier(random_state=0).fit(Xtr, ytr)
rf = RandomForestClassifier(
    n_estimators=300, oob_score=True, random_state=0
).fit(Xtr, ytr)
print(f"Single deep tree AUC: {roc_auc_score(yte, tree.predict_proba(Xte)[:,1]):.3f}")
print(f"Random forest AUC:    {roc_auc_score(yte, rf.predict_proba(Xte)[:,1]):.3f}")
print(f"Random forest OOB AUC:{roc_auc_score(ytr, rf.oob_decision_function_[:,1]):.3f}")
# Expected: single deep tree AUC ~0.56, forest ~0.69. Averaging randomized trees
#           improves ranking in this one sample; variance requires repeated fits.

#### Your research-record entry

*Double-click this cell to edit it, then press Shift-Enter to render.*

| FIELD | YOUR ENTRY |
|:--|:--|
| fixed comparison | `[sample, split, and common features]` |
| models | `[single-tree and forest specifications]` |
| evidence | `[tree held-out AUC plus forest OOB and held-out AUC]` |
| mechanism | `[why averaging can reduce instability]` |
| narrow claim | `[what this held-out comparison supports]` |
| limitation | `[what it cannot establish universally]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

**Completed forest entry**

| FIELD | EXAMPLE ENTRY |
|:--|:--|
| fixed comparison | 3,000 simulated rows and six features; first 1,500 rows train both models and the final 1,500 test them |
| models | one unrestricted classification tree versus a 300-tree random forest |
| evidence | single-tree held-out AUC **0.560**; forest OOB AUC **0.670**; forest held-out AUC **0.693** |
| mechanism | bootstrap samples and random feature subsets make the trees disagree; averaging reduces their unstable errors |
| narrow claim | on this one held-out sample, averaging many randomized trees improved ranking over one deep tree |
| limitation | the result does not prove that forests outperform every tuned tree or every simpler baseline on financial data |

The independent comparison, rather than the number of fitted trees, supports the
narrow result.

</details>

### Demo 5 — Impurity vs. permutation importance

> **Break cue:** Deck A, after recording segment S3. Read the task on the preceding slide, predict what the output should show, then run the cell. Fill in the **research-record entry** below the output — the fields are named for you, and a completed example sits under it once you have written yours.

### Remove a clue and see what changes

`feature_importances_` allocates total training impurity reduction across inputs; the shares sum to 1. A many-valued noise feature offers many possible cuts. `permutation_importance(..., scoring="accuracy")` holds the fitted forest fixed, shuffles one holdout column, and measures the **decrease in accuracy**. Its ten shuffles summarize permutation variation, not uncertainty across new training samples.

The second comparison uses **AUC decrease**, not accuracy decrease. `changed[:, columns] = changed[permutation][:, columns]` applies one shared row permutation to a group, preserving the relationship between its columns while breaking their alignment with the outcome. This small example uses one seeded shuffle; repeated shuffles would quantify its variability. Impurity shares, accuracy drops, and AUC drops have different scales.

**Predict, then check:** if two columns carry almost the same clue, what happens when only one is shuffled? Explain why reliance is not a causal effect and why a near-zero individual score does not prove a correlated feature is useless.

In [ ]:
# Three real signals plus low- and high-cardinality noise features.
N = 4000
real_1 = rng.standard_normal(N)
real_2 = rng.standard_normal(N)
real_3 = rng.standard_normal(N)
noise_binary = rng.integers(0, 2, N)
noise_many_values = rng.permutation(np.arange(N))
X = np.column_stack([real_1, real_2, real_3, noise_binary, noise_many_values])
logits = 0.7 * X[:, 0] + 0.5 * X[:, 1] - 0.3 * X[:, 2] + rng.standard_normal(N) * 1.0
y = (logits > 0).astype(int)
ntr = N // 2
Xtr, Xte, ytr, yte = X[:ntr], X[ntr:], y[:ntr], y[ntr:]
rf = RandomForestClassifier(n_estimators=200, random_state=0).fit(Xtr, ytr)

imp = rf.feature_importances_
perm = permutation_importance(rf, Xte, yte, n_repeats=10, random_state=0, scoring="accuracy").importances_mean

names = ["real_1", "real_2", "real_3", "noise_binary", "noise_many_values"]
print("Impurity importances:")
for n, v in zip(names, imp): print(f"  {n:20s} {v:.4f}")
print("Permutation importances (held-out accuracy decrease):")
for n, v in zip(names, perm): print(f"  {n:20s} {v:.4f}")
print("Noise-feature cardinalities in training:")
print(f"  noise_binary={np.unique(Xtr[:, 3]).size}")
print(f"  noise_many_values={np.unique(Xtr[:, 4]).size}")

# A second known-truth comparison: two correlated columns carry one idea.
X_group = np.column_stack([
    real_1,
    real_1 + rng.standard_normal(N) * 0.08,
    real_2,
    real_3,
])
Xgtr, Xgte = X_group[:ntr], X_group[ntr:]
rf_group = RandomForestClassifier(n_estimators=200, random_state=0).fit(Xgtr, ytr)
base_auc = roc_auc_score(yte, rf_group.predict_proba(Xgte)[:, 1])
group_rng = np.random.default_rng(7)
permutation = group_rng.permutation(len(Xgte))

def auc_drop_when_permuted(columns):
    changed = Xgte.copy()
    changed[:, columns] = changed[permutation][:, columns]
    return base_auc - roc_auc_score(yte, rf_group.predict_proba(changed)[:, 1])

drop_a = auc_drop_when_permuted([0])
drop_b = auc_drop_when_permuted([1])
drop_group = auc_drop_when_permuted([0, 1])
print("Correlated-signal AUC drops:")
print(f"  permute signal_a alone: {drop_a:.4f}")
print(f"  permute signal_b alone: {drop_b:.4f}")
print(f"  permute both together:  {drop_group:.4f}")
# Expected: training impurity gives many-valued noise more opportunities than
# binary noise. Held-out permutation rejects both; grouping correlated signals
# reveals reliance that either individual permutation can understate.

#### Your research-record entry

*Double-click this cell to edit it, then press Shift-Enter to render.*

| FIELD | YOUR ENTRY |
|:--|:--|
| known truth | `[real, low-cardinality noise, many-valued noise, and correlated substitute features]` |
| training importance | `[both noise features' cardinality and impurity importance]` |
| held-out counterfactual | `[noise permutation values plus individual and grouped correlated-signal AUC decreases]` |
| interpretation | `[question each importance measure asks]` |
| narrow claim | `[what the controlled comparison supports]` |
| limitation | `[why importance is not a causal or unique attribution]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

**Completed importance-audit entry**

| FIELD | EXAMPLE ENTRY |
|:--|:--|
| known truth | three features affect the outcome; binary and many-valued noise are useless; a second experiment gives one signal two correlated columns |
| training importance | impurity assigns binary noise **0.0218** and many-valued noise **0.1672** despite both being useless |
| held-out counterfactual | accuracy decreases are near zero for both noise features; permuting correlated A or B lowers AUC **0.0813/0.0527**, while permuting both lowers it **0.1892** |
| interpretation | the training split ledger rewards opportunities to reduce impurity; held-out permutation asks whether breaking a feature damages prediction |
| narrow claim | in this controlled example, held-out permutation recovers the planted distinction more faithfully |
| limitation | correlated substitutes can split or hide importance, so an individual permutation score is not a causal effect |

An importance number is meaningful only beside the population and
counterfactual that produced it.

</details>

### Mini-project: audit a forest

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

Run **Demos 4–5** in `week5_demos.ipynb`.

1. Compare OOB and held-out area under the receiver-operating-characteristic curve (AUC), a ranking score.
2. Add a high-cardinality noise feature.
3. Compare impurity and held-out permutation importance.
4. Group two correlated signal features and permute them together.


Your output is one diagnostic table, not another model.

**Research record:** the OOB estimate beside the independent-holdout score; the noise
feature's cardinality; impurity against held-out permutation importance; the
grouped-permutation result.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 6 — Gradient boosting: stagewise build, learning rate, early stopping

> **Break cue:** Deck B, after recording segment S4. Read the task on the preceding slide, predict what the output should show, then run the cell. Fill in the **research-record entry** below the output — the fields are named for you, and a completed example sits under it once you have written yours.

### Watch the partial prediction grow

`GradientBoostingRegressor` starts from a constant prediction, normally the training mean under squared-error loss. Each new tree fits the remaining error; `learning_rate` scales its contribution. `staged_predict(Xva)` yields validation predictions after 1 tree, 2 trees, and so on. These are checkpoints along one fit, not independent fits.

The vertical score is `1 - model_squared_error / baseline_squared_error`, where the baseline predicts the **frozen training mean** for every validation case. Zero ties that feasible forecast. Positive values improve on it. The horizontal axis starts at stage 1. All three rates get the same 500-stage cap, depth, data, and score.

**Predict, then check:** with current prediction 2, observed outcome 5, and a fitted residual of 3, a learning rate of 0.1 changes the prediction to `2 + 0.1 * 3 = 2.3`. Why might a smaller rate need more stages? See the [official worked boosting example](https://scikit-learn.org/stable/auto_examples/ensemble/plot_gradient_boosting_regression.html) for another validation path; it uses different data.

In [ ]:
N = 4000
X = rng.standard_normal((N, 8))
y = (0.5 * X[:, 0] * X[:, 1] - 0.3 * np.abs(X[:, 2])
     + 0.2 * X[:, 3] - 0.2 * X[:, 4] + rng.standard_normal(N) * 0.8)
ntr, nval = 1500, 1000
Xtr, Xva, Xte = X[:ntr], X[ntr:ntr+nval], X[ntr+nval:]
ytr, yva, yte = y[:ntr], y[ntr:ntr+nval], y[ntr+nval:]

# Three learning rates
plt.figure(figsize=(8, 4))
boosting_path_rows = []
for eta, color in [(0.01, "blue"), (0.05, "green"), (0.3, "red")]:
    gbm = GradientBoostingRegressor(n_estimators=500, learning_rate=eta,
                                    max_depth=3, random_state=0).fit(Xtr, ytr)
    val_r2 = []
    for pred in gbm.staged_predict(Xva):
        ss_res = ((yva - pred) ** 2).sum()
        ss_tot = ((yva - ytr.mean()) ** 2).sum()  # feasible training-mean forecast
        val_r2.append(1 - ss_res / ss_tot)
    best_stage = int(np.argmax(val_r2)) + 1
    boosting_path_rows.append({
        "learning_rate": eta,
        "best_stage": best_stage,
        "best_validation_R2": max(val_r2),
        "stage_500_R2": val_r2[-1],
    })
    plt.plot(np.arange(1, 501), val_r2, color=color, label=f"η={eta}")
plt.axhline(0, color="k", lw=0.5)
plt.xlabel("number of trees"); plt.ylabel("validation skill vs. training mean")
plt.title("Learning rate / n-trees tradeoff")
plt.legend(); plt.tight_layout()
save_lecture_figure(plt.gcf(), "demo6_boosting_paths.png")
plt.show()
print(pd.DataFrame(boosting_path_rows).round(4).to_string(index=False))
# Expected: a smaller learning rate needs more stages to reach the same loss. The path,
#           not the cap, decides the fitted function.

#### Your research-record entry

*Double-click this cell to edit it, then press Shift-Enter to render.*

| FIELD | YOUR ENTRY |
|:--|:--|
| design | `[sample and training/validation/test roles]` |
| fixed components | `[loss, depth, cap, and rates compared]` |
| path evidence | `[how learning rate changes the path]` |
| selected object | `[what would be chosen on validation]` |
| narrow claim | `[what rate and stage count jointly control]` |
| limitation | `[what validation has not yet tested]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

**Completed boosting-path entry**

| FIELD | EXAMPLE ENTRY |
|:--|:--|
| design | 4,000 simulated rows and eight features; 1,500 train, the next 1,000 validate, and 1,500 remain for independent test |
| fixed components | squared-error boosting, depth-3 trees, 500-stage cap, and learning rates 0.01, 0.05, and 0.30 |
| path evidence | best stages are **500**, **386**, and **43** for rates 0.01, 0.05, and 0.30; their best validation skill values against the frozen training mean are **0.1669**, **0.2497**, and **0.2699** |
| selected object | a learning-rate-and-stage pair chosen from the validation path, not automatically the 500-tree endpoint |
| narrow claim | learning rate and number of stages jointly control the fitted additive function |
| limitation | this path selects among three declared rates in one simulation; it does not test the selected procedure on the final block |

The cap says how far the search may run. The validation path says where the
procedure should stop.

</details>

### Demo 7 — Early stopping: find the round that minimizes validation MSE

> **Break cue:** Deck B, after recording segment S4. Read the task on the preceding slide, predict what the output should show, then run the cell. Fill in the **research-record entry** below the output — the fields are named for you, and a completed example sits under it once you have written yours.

### Select a checkpoint, not a test result

Run Demo 6 first. `staged_predict` supplies every partial ensemble's predictions. `((yva - p) ** 2).mean()` is mean squared error (MSE), so lower is better. `np.argmin(val_mse) + 1` selects its earliest minimum. The example fits all 500 stages and selects retrospectively on validation; it does not save computation through a patience-based stop.

The chosen `best_round` belongs to model development. Demo 9 will refit at that stage count and open a separate test. These rows are independent simulation draws; for actual future forecasts the validation rows must also be later in time, with mature labels and training-only preprocessing.

**Predict, then check:** if the minimum occurs at stage 500, have we shown that 500 is optimal? Explain why that is a search-boundary warning.

In [ ]:
gbm = GradientBoostingRegressor(n_estimators=500, learning_rate=0.05,
                                max_depth=3, random_state=0).fit(Xtr, ytr)
val_mse = [((yva - p) ** 2).mean() for p in gbm.staged_predict(Xva)]
best_round = int(np.argmin(val_mse)) + 1
print(f"Validation MSE minimum at round {best_round}/500")
print(f"MSE at best round: {min(val_mse):.4f}")
print(f"MSE at round 500:  {val_mse[-1]:.4f}  (worse — past the minimum)")

plt.figure(figsize=(7, 3.5))
plt.plot(np.arange(1, 501), val_mse)
plt.axvline(best_round, ls="--", color="red", label=f"min at {best_round}")
plt.xlabel("number of trees"); plt.ylabel("validation MSE")
plt.title("Early stopping: the bottom of the curve, not the endpoint")
plt.legend(); plt.tight_layout()
save_lecture_figure(plt.gcf(), "demo7_early_stopping.png")
plt.show()

#### Your research-record entry

*Double-click this cell to edit it, then press Shift-Enter to render.*

| FIELD | YOUR ENTRY |
|:--|:--|
| boundary | `[fit and validation rows]` |
| fixed candidate | `[learning rate, depth, and stage cap]` |
| selected round | `[validation-minimizing round]` |
| evidence | `[MSE at selected round and cap]` |
| narrow claim | `[what the loss path supports]` |
| limitation | `[why another later block is still needed]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

**Completed early-stopping entry**

| FIELD | EXAMPLE ENTRY |
|:--|:--|
| boundary | fit on the first 1,500 rows and inspect loss only on the following 1,000-row validation block |
| fixed candidate | learning rate 0.05, depth 3, and a maximum of 500 boosting stages |
| selected round | validation MSE reaches its minimum at round **386** |
| evidence | MSE **0.7337** at round 386 versus **0.7352** at round 500 |
| narrow claim | for this fitted path, continuing to the cap slightly worsened validation loss |
| limitation | round 386 is now selected; a separate later block is still required to test the complete selection procedure |

Early stopping uses validation data to choose a model. It does not turn that
same validation loss into final evidence.

</details>

### Mini-project: trace the boosting path

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

Run **Demos 6–7** in `week5_demos.ipynb`.

1. Start from the training mean and fit a sequence of tree corrections.
2. Plot the full validation path for three learning rates.
3. Record the best stage for each rate and the value at the 500-stage cap.
4. For learning rate 0.05, compare the selected stage with the cap.


Explain the path, not merely the last score.

**Research record:** the starting value; depth and stage cap; all learning
rates; the selected stage and validation result for each path.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Real-data companion — linear, forest, and boosting forecasts on one market sample

> **Optional empirical comparison:** Deck B, after recording segment S4. Run this after the known-truth demo. Use the same transformation and compare the simulation with the historical series. Record one important difference and the most likely reason for it.

### Apply the comparison to a dated market problem

`rolling(21).mean()` uses the current and previous 20 returns, available after today's close. `shift(-1)` puts tomorrow's return on today's feature row: it creates the **target**, never a predictor. `dropna()` removes incomplete windows and the unavailable final target. The 60/20/20 split follows dates.

Each candidate's `.fit(...)` uses only the first block; validation MSE picks one candidate per family. Here we deliberately keep that training fit after selection, with no development refit. All families face the same final dates and training-mean baseline. `test_R2` is squared-error skill against that frozen mean; `rank_corr` is Spearman correlation across market dates, not a same-date cross-sectional stock-ranking score.

This companion forecasts **continuous Fama–French market returns** on one split. The slides' empirical figures forecast **S&P 500 direction probabilities** across five outer periods and use Brier skill. The companion illustrates a common protocol; it does not reproduce those figures. The bundled snapshot reproduces the printed dates; downloading the public fallback can change them.

**Predict, then check:** would adding tomorrow's return as an input make the comparison invalid even if all models used it? Record the actual train, validation, and test dates printed by this run.

In [ ]:
ff_models_raw, ff_models_source = course_csv("datasets/famafrench/ff_factors_daily.csv")
ff_models_raw["date"] = pd.to_datetime(ff_models_raw["date"].astype(str), format="%Y%m%d")
ff_models = ff_models_raw.set_index("date").sort_index().loc["1990":]
market_models_return = (ff_models["Mkt-RF"] + ff_models["RF"]) / 100.0
model_frame = pd.DataFrame({
    "return_t": market_models_return,
    "mom5": market_models_return.rolling(5).mean(),
    "mom21": market_models_return.rolling(21).mean(),
    "mom63": market_models_return.rolling(63).mean(),
    "vol5": market_models_return.rolling(5).std(),
    "vol21": market_models_return.rolling(21).std(),
    "vol63": market_models_return.rolling(63).std(),
    "next_return": market_models_return.shift(-1),
}).dropna()
real_model_columns = [column for column in model_frame.columns if column != "next_return"]
real_model_train_end = int(0.6 * len(model_frame))
real_model_validation_end = int(0.8 * len(model_frame))
real_model_train = model_frame.iloc[:real_model_train_end]
real_model_validation = model_frame.iloc[real_model_train_end:real_model_validation_end]
real_model_test = model_frame.iloc[real_model_validation_end:]

# Two validation choices per flexible family; the test block is opened once.
real_candidates = {
    "linear": [LinearRegression()],
    "forest": [
        RandomForestRegressor(n_estimators=250, max_depth=depth, min_samples_leaf=20, random_state=0, n_jobs=-1)
        for depth in (3, 6)
    ],
    "boosting": [
        GradientBoostingRegressor(n_estimators=250, learning_rate=0.03, max_depth=depth, random_state=0)
        for depth in (1, 2)
    ],
}
real_selected = {}
for family, candidates in real_candidates.items():
    validation_mse = []
    for candidate in candidates:
        candidate.fit(real_model_train[real_model_columns], real_model_train["next_return"])
        validation_mse.append(np.mean(
            (real_model_validation["next_return"] - candidate.predict(real_model_validation[real_model_columns])) ** 2
        ))
    real_selected[family] = candidates[int(np.argmin(validation_mse))]

real_model_rows = []
baseline_prediction = np.repeat(real_model_train["next_return"].mean(), len(real_model_test))
baseline_mse = np.mean((real_model_test["next_return"] - baseline_prediction) ** 2)
real_model_rows.append({"model": "training mean", "test_R2": 0.0, "rank_corr": np.nan})
for family, model in real_selected.items():
    prediction = model.predict(real_model_test[real_model_columns])
    test_mse = np.mean((real_model_test["next_return"] - prediction) ** 2)
    rank_corr = pd.Series(prediction).corr(
        pd.Series(real_model_test["next_return"].to_numpy()), method="spearman"
    )
    real_model_rows.append({"model": family, "test_R2": 1 - test_mse / baseline_mse, "rank_corr": rank_corr})

print("Source: Kenneth R. French Data Library, bundled daily factors")
print(f"File: {ff_models_source}")
print(f"Train/validation/test end dates: {real_model_train.index.max().date()}, "
      f"{real_model_validation.index.max().date()}, {real_model_test.index.max().date()}")
print("Target: next-day decimal market return; all features available after close t")
print(pd.DataFrame(real_model_rows).round(4).to_string(index=False))
print("The same frozen test block evaluates every model family.")

#### Your research-record entry

*Double-click this cell to edit it, then press Shift-Enter to render.*

| FIELD | YOUR ENTRY |
|:--|:--|
| source and target | `[dataset, horizon, and units]` |
| clock and boundary | `[availability rule plus train/validation/test dates]` |
| search | `[candidate models and validation choices]` |
| frozen test evidence | `[predictive R² and rank correlation for each family]` |
| narrow claim | `[what this one comparison supports]` |
| limitation | `[why this cannot rank families universally]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

**Completed optional market entry**

| FIELD | EXAMPLE ENTRY |
|:--|:--|
| source and target | Kenneth R. French daily US market factors; next-day decimal market return |
| clock and boundary | features available after close *t*; train ends 2011-11-10, validation ends 2019-02-04, test ends 2026-04-29 |
| search | linear regression; forest depths 3 or 6; boosting depths 1 or 2, with flexible-family choice made on validation |
| frozen test evidence | linear predictive R² **+0.0010** and rank correlation **+0.0271**; forest **−0.0261/−0.0273**; boosting **−0.0601/+0.0079** |
| narrow claim | in this one frozen market block and feature set, flexibility did not improve on the training-mean baseline; the line barely did in squared loss |
| limitation | one market, horizon, feature set, search, and test period cannot rank these model families universally |

The richer model gets no presumption of superiority. The same feasible baseline
and untouched rows evaluate every family.

</details>

### Demo 8 — The leakage amplifier

> **Break cue:** Deck B, after recording segment S5. Read the task on the preceding slide, predict what the output should show, then run the cell. Fill in the **research-record entry** below the output — the fields are named for you, and a completed example sits under it once you have written yours.

### Diagnose a field that gives away the answer

This cell generates its own `audit_X, audit_y` using a separate seed. It never scores or alters Demo 9's test rows. `leak = audit_y + noise` is an intentionally impossible input: it depends on the outcome we are trying to predict. `np.column_stack` appends it as feature 8, the ninth column because indexing starts at zero.

Ridge, forest, and boosting each fit clean and contaminated versions of the same audit population. `spearmanr(...).statistic` compares the ordering of predictions and outcomes. Permutation uses `scoring="neg_mean_squared_error"`, so its reported score decrease equals an **increase in MSE**. That is a different quantity from the table's rank correlation.

**Predict, then check:** can the linear model exploit the leak too? After locating feature 8, explain why its creation from the future outcome—not its large importance—establishes the timing violation.

In [ ]:
# A separate audit population keeps Demo 9's test sample unopened.
audit_rng = np.random.default_rng(508)
audit_X = audit_rng.standard_normal((4000, 8))
audit_y = (0.5 * audit_X[:, 0] * audit_X[:, 1] - 0.3 * np.abs(audit_X[:, 2])
           + 0.2 * audit_X[:, 3] - 0.2 * audit_X[:, 4]
           + audit_rng.standard_normal(4000) * 0.8)
audit_Xtr, audit_Xte = audit_X[:1500], audit_X[2500:]
audit_ytr, audit_yte = audit_y[:1500], audit_y[2500:]
# Build a noisy copy of the audit target itself as an inadmissible "feature".
leak = audit_y + audit_rng.standard_normal(len(audit_y)) * 0.3
X_leak = np.column_stack([audit_X, leak])
Xtr_l, Xte_l = X_leak[:1500], X_leak[2500:]

leak_models = {
    "Ridge": Ridge(alpha=1.0),
    "forest": RandomForestRegressor(
        n_estimators=200, max_depth=6, min_samples_leaf=10,
        random_state=0, n_jobs=-1,
    ),
    "boosting": GradientBoostingRegressor(
        n_estimators=200, learning_rate=0.05, max_depth=3, random_state=0,
    ),
}
leak_rows = []
leaked_booster = None
for name, estimator in leak_models.items():
    admissible = estimator.fit(audit_Xtr, audit_ytr)
    admissible_rank_ic = spearmanr(admissible.predict(audit_Xte), audit_yte).statistic
    leaked = estimator.__class__(**estimator.get_params()).fit(Xtr_l, audit_ytr)
    leaked_rank_ic = spearmanr(leaked.predict(Xte_l), audit_yte).statistic
    leak_rows.append({
        "model": name,
        "admissible_rank_ic": admissible_rank_ic,
        "target_proxy_rank_ic": leaked_rank_ic,
    })
    if name == "boosting":
        leaked_booster = leaked

print(pd.DataFrame(leak_rows).round(4).to_string(index=False))
leak_permutation = permutation_importance(
    leaked_booster, Xte_l, audit_yte, scoring="neg_mean_squared_error",
    n_repeats=5, random_state=0,
).importances_mean
print(f"Largest held-out permutation loss increase: feature {int(np.argmax(leak_permutation))} "
      f"({max(leak_permutation):.4f}); planted proxy is feature {X_leak.shape[1] - 1}")
# Expected: all three learners exploit the target proxy, and held-out permutation
#           points to the planted final column. Lineage, not importance alone,
#           establishes that the feature was inadmissible.

#### Your research-record entry

*Double-click this cell to edit it, then press Shift-Enter to render.*

| FIELD | YOUR ENTRY |
|:--|:--|
| planted violation | `[inadmissible feature and how it is built]` |
| fixed comparison | `[model, rows, and settings held fixed]` |
| evidence | `[admissible and target-proxy later rank ICs]` |
| information path | `[how the outcome crossed into the feature]` |
| audit response | `[specific check and action]` |
| narrow claim | `[what the contrast demonstrates]` |
| limitation | `[what this obvious planted case cannot establish]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

**Completed leakage entry**

| FIELD | EXAMPLE ENTRY |
|:--|:--|
| planted violation | a noisy copy of the outcome in a separate seeded audit population is appended as a feature |
| fixed comparison | Ridge, forest, and boosting use the same admissible or contaminated feature tables and the same independent audit rows |
| evidence | admissible-feature rank IC is **0.2383/0.3456/0.4048** for Ridge/forest/boosting; with the proxy it becomes **0.9550/0.9529/0.9533**; held-out permutation identifies planted feature 8 |
| information path | the proxy is computed from the outcome, so the later label crosses into the feature row before fitting |
| audit response | reject the feature by tracing its source and latest timestamp; do not celebrate or merely regularize the score |
| narrow claim | every model family, linear included, turned the same target proxy into spectacular but invalid apparent performance |
| limitation | the intentionally obvious proxy does not measure how easy every real leak will be to find |

Leakage is a data-lineage failure. Model tuning cannot repair an inadmissible
feature.

</details>

### Mini-project: plant and find a leak

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

Run **Demo 8** in `week5_demos.ipynb`.

Add one disguised target proxy to the training table. Compare Ridge,
forest, and boosting. Then locate the problem with a field-creation audit
and held-out permutation.

**Research record:** the planted proxy and how it was disguised; each model's
response to it; the audit step that located it.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 9 — Freeze the stopping rule, then test once

> **Break cue:** Deck B, after recording segment S6. Read the task on the preceding slide, predict what the output should show, then run the cell. Fill in the **research-record entry** below the output — the fields are named for you, and a completed example sits under it once you have written yours.

### Freeze, refit, and open the final test

Run Demos 6 and 7 first. `best_round` fixes the stage count before this cell scores the test. `np.vstack([Xtr, Xva])` and `np.concatenate([ytr, yva])` pool development data. The selected model is **refitted** on those rows at the frozen rate, depth, and stage count.

The 500-stage model is a **prespecified comparator**, not another candidate selected with test results. Both face the same final rows. `y_development.mean()` is a feasible constant forecast. `1 - selected_test_mse / baseline_test_mse` reports improvement against it. The audit sample in Demo 8 is separate, so this is the first opening of these test labels for evaluation.

**Predict, then check:** write the chosen stage before running this cell. If the full-cap comparator wins, explain why switching to it would consume this test as development evidence for the revised rule.

In [ ]:
X_development = np.vstack([Xtr, Xva])
y_development = np.concatenate([ytr, yva])
selected_booster = GradientBoostingRegressor(
    n_estimators=best_round, learning_rate=0.05, max_depth=3, random_state=0,
).fit(X_development, y_development)
full_cap_booster = GradientBoostingRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=3, random_state=0,
).fit(X_development, y_development)

selected_test_mse = np.mean((yte - selected_booster.predict(Xte)) ** 2)
full_cap_test_mse = np.mean((yte - full_cap_booster.predict(Xte)) ** 2)
baseline_test_mse = np.mean((yte - y_development.mean()) ** 2)
selected_test_r2 = 1 - selected_test_mse / baseline_test_mse
print(f"Validation-selected stage: {best_round}")
print(f"Frozen selected-stage test MSE: {selected_test_mse:.4f}")
print(f"Full-cap test MSE:             {full_cap_test_mse:.4f}")
print(f"Frozen development-mean MSE:        {baseline_test_mse:.4f}")
print(f"Selected-stage predictive R²:       {selected_test_r2:+.4f}")
# Expected: the test block evaluates the stage selected earlier. Its result
#           must not be used to choose a new stage without changing the block's role.

#### Your research-record entry

*Double-click this cell to edit it, then press Shift-Enter to render.*

| FIELD | YOUR ENTRY |
|:--|:--|
| boundary | `[training, validation, and later test rows]` |
| selected procedure | `[rate, depth, cap, and validation-selected stage]` |
| refit rule | `[eligible rows and fixed stage count]` |
| later evidence | `[selected-stage, full-cap, and baseline MSE plus predictive R²]` |
| narrow claim | `[what the frozen later comparison supports]` |
| limitation | `[why the later result cannot retune the rule and remain independent test evidence]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

**Completed frozen-stopping entry**

| FIELD | EXAMPLE ENTRY |
|:--|:--|
| boundary | 1,500 rows train, 1,000 independent rows select the stage, and 1,500 reserved rows test it once |
| selected procedure | learning rate **0.05**, depth **3**, 500-stage cap, and validation-selected stage **386** |
| refit rule | refit through the validation boundary using the fixed 386-stage count; do not monitor the test block |
| later evidence | selected-stage MSE **0.7904**; full-cap MSE **0.7724**; frozen development-mean MSE **1.0532**; selected-stage predictive R² **+0.2496** |
| narrow claim | the frozen selected procedure improved on the development-mean baseline in this controlled independent block |
| limitation | the full cap happened to perform better later; using that fact to change the stopping rule makes this block development data for the revision |

The later block evaluates the rule that was selected earlier. It does not
retroactively select a different stopping stage.

</details>

### Mini-project: select the stopping stage before testing

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

Run **Demo 9** in `week5_demos.ipynb`.

1. Reserve the last block and do not plot it.
2. Choose the boosting stage on the preceding block.
3. Refit on training plus validation rows at that stage.
4. Reveal the last block once.
5. Record both the selected round and the final loss.

**Research record:** the reserved block; the stage chosen on the preceding
block; the refit rule; the selected round and the final loss, revealed once.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Simulated companion — credit scoring on held-out borrowers

> **Break cue:** Deck B, after recording segment S6. Read the task on the preceding slide, predict what the output should show, then run the cell. Fill in the **research-record entry** below the output — the fields are named for you, and a completed example sits under it once you have written yours.

### Compare ranking with probability quality

The generator creates debt, utilization, liquidity, and delinquency inputs. Its known `log_odds` include two threshold interactions. `1 / (1 + exp(-log_odds))` turns that score into a probability; comparing a uniform random draw with it produces a simulated default event.

The first 4,200 independent borrowers train both models; the other 1,800 test them. Logistic regression adds contributions on the log-odds scale, while the forest can learn conditional thresholds. `.predict_proba(...)[:, 1]` supplies default probabilities. AUC measures ranking; Brier loss is the mean squared probability error, with lower being better. Neither alone establishes good calibration or a profitable lending rule.

**Predict, then check:** could always predicting “no default” have high accuracy and still be unhelpful? Record the holdout default rate and explain why these simulated borrowers do not establish performance on a real later cohort.

In [ ]:
# The known truth contains two threshold interactions. Logistic regression and
# the forest receive identical features and independent simulated borrowers.
credit_rng = np.random.default_rng(510)
N = 6000
credit = pd.DataFrame({
    "debt_to_income": credit_rng.beta(2.5, 4.0, N),
    "utilization": credit_rng.beta(2.0, 2.5, N),
    "liquidity": credit_rng.beta(2.0, 5.0, N),
    "delinquencies": credit_rng.poisson(0.35, N),
})
log_odds = (
    -4.2
    + 1.8 * credit["debt_to_income"].to_numpy()
    + 1.2 * credit["utilization"].to_numpy()
    - 1.5 * credit["liquidity"].to_numpy()
    + 0.35 * credit["delinquencies"].to_numpy()
    + 1.5 * (
        (credit["debt_to_income"].to_numpy() > 0.55)
        & (credit["utilization"].to_numpy() > 0.75)
    )
    + 1.0 * (
        (credit["delinquencies"].to_numpy() >= 2)
        & (credit["liquidity"].to_numpy() < 0.15)
    )
)
default_probability = 1 / (1 + np.exp(-log_odds))
default = (credit_rng.uniform(size=N) < default_probability).astype(int)

cohort_cut = 4200
X_credit = credit.to_numpy()
X_credit_train, X_credit_test = (
    X_credit[:cohort_cut],
    X_credit[cohort_cut:],
)
y_credit_train, y_credit_test = default[:cohort_cut], default[cohort_cut:]

credit_logit = LogisticRegression(max_iter=1000).fit(
    X_credit_train, y_credit_train
)
credit_forest = RandomForestClassifier(
    n_estimators=400,
    max_depth=6,
    min_samples_leaf=30,
    random_state=0,
).fit(X_credit_train, y_credit_train)

print("\nSimulated held-out credit comparison (same rows and features):")
for name, model in [
    ("logistic baseline", credit_logit),
    ("random forest", credit_forest),
]:
    probability = model.predict_proba(X_credit_test)[:, 1]
    print(
        f"  {name:17s} "
        f"AUROC={roc_auc_score(y_credit_test, probability):.3f}  "
        f"Brier={brier_score_loss(y_credit_test, probability):.4f}"
    )
print(f"  held-out default rate={y_credit_test.mean():.3f}")
# One simulated holdout cannot establish calibration stability or lending value.

#### Your research-record entry

*Double-click this cell to edit it, then press Shift-Enter to render.*

| FIELD | YOUR ENTRY |
|:--|:--|
| target and population | `[event, simulated training and holdout borrowers, and prevalence]` |
| fixed information | `[features shared by both models]` |
| model capacity | `[what structure each family can represent]` |
| later evidence | `[AUC and Brier for both models]` |
| narrow claim | `[what the later comparison supports]` |
| limitation | `[what the simulation and metrics cannot establish]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

**Completed simulated-credit entry**

| FIELD | EXAMPLE ENTRY |
|:--|:--|
| target and population | synthetic default event; 4,200 independent rows train and 1,800 held-out rows test both models; holdout default rate **0.053** |
| fixed information | debt-to-income, utilization, liquidity, and delinquency count for both logistic regression and the forest |
| model capacity | logistic regression supplies additive log-odds; the depth-6 forest can represent threshold interactions |
| later evidence | logistic AUC **0.709**, Brier **0.0478**; forest AUC **0.647**, Brier **0.0485** |
| narrow claim | on this one simulated holdout, logistic regression had better ranking and lower Brier loss |
| limitation | the interactions were planted; the rows are independent and carry no dates; this does not establish deployment value, calibration stability, or fairness |

The comparison is interpretable because the models receive the same information
and the same independent holdout.

</details>

## Where this leaves us

Flexible learners can uncover useful nonlinear structure, but they can also magnify noise, leakage, and researcher choice. The object tested on later data is therefore the whole selection procedure, not merely a final fitted forest or booster.

Week 6 asks how small forecast improvements accumulate across opportunities, how costs and search reduce the apparent edge, and how uncertainty should limit the claim.